# 🍫 **Chocolate Recognition (Deep Learning)**  
### *Final IAPR 2025 Project – Group 14 – Niels Rigaud, Caspar Henking, Aksel Acar*

---

## Table of Contents
- [Introduction](#introduction)
- [Methodology](#methodology)
  - [Dataset](#dataset)
  - [Data Augmentation](#data-augmentation)
  - [YOLO: main concepts](#YOLO:main_concepts)
  - [Model](#model)
  - [Loss](#loss)
  - [Post-processing](#post_processing)
  - [Training](#training)
- [Results & Conclusion](#results)

---


# Introduction

The goal of the project was to implement a **chocolate detector** that's able to count the number of chocolates of predefined types in an image. The main challenge lies in the presence of 13 different types of chocolates, combined with images that feature a wide variety of backgrounds, both of which significantly complicate the detection task. The template of each of the 13 different chocolates was given beforehand.

For this purpose, our group created a neural network on the basis of the **YOLO architecture** (You Only Look Once). This model is designed not only to detect and classify multiple chocolates within an image, but also to localize them using bounding boxes (details are provided in the dedicated methodology section).

# Methodology

## Dataset:

The furnished dataset consisted of 13 reference images (for the 13 classes), with one chocolate by image on a blank background.
The training set consisted of 90 images featuring chocolates against various backgrounds, including non-chocolate objects and partially occluded chocolates. The test set included 180 images of the same type as the train set. It is important to note that every image is 6000x4000 pixels.

Additionally, the ground truth was given in a CSV file, each row corresponding to an image of the train set, and the 13 last columns corresponding to the number of chocolates of each type in the associated image. However, we decided to use a YOLO algorithm, that requires bounding boxes for each chocolates, associated with the right label. Therefore, we used the tool VGG Image Annotator (https://www.robots.ox.ac.uk/~vgg/software/via/) to create the bounding boxes.

A slight reshaping of the created CSV was necessary in order to keep only the relevant columns. Each row corresponds to a single chocolate: the first column being the name of the picture, followed by the center coordinates cx and cy, then the width and height of the bounding box and finally the chocolate class with an integer between 0 and 12 coresponding to the 13 classes. All values are in pixel, with respect to the original images (6000x4000). We additionally had to shift cx and cy because by default VGG image Annotator gives the upper-left corner of the box, whereas the convention for YOLO is the center. The final labels can be found in the file `src/Annotations_and_labels/raw_annotations_YOLO.csv`.

Finally, the file `src/Annotations_and_labels/Create_labels_and_link_to_images.py` has two tasks. First, it creates a folder `src/Annotations_and_labels/labels` inside which there is one file per training image. Each file includes one row per chocolate with all coordinates (center, width and height of each chocolate box) normalized between 0 and 1, as required by the YOLO format. The second task is to create a mapping between the images names (.JPG) and their labels (.txt). This is more convenient for the rest of the code. The result generates a CSV file `src/Annotations_and_labels/link_images_labels_YOLO.csv` with two columns, one with a `.JPG`, the other with a `.txt`, with each row corresponding to one image name.

## Data augmentation

In our project the training dataset consisted of 90 images along with 13 reference images, each showing a single type of chocolate on a plain white background. We decided to combine the reference images with the main training datatset resulting in 103 images to train our model on. Naturally, from the now 103 train images we had to create a validation set in order to track the performance of our model on unseen data, checking for overfitting. However, due to the limited number of availale images in the train set, we selected the strict minimum number of images for  validation. We chose one image per noisy background and 3 of neutral background (more frequent). This resulted in 95 train images and 8 validation images. 

Once the train set and validation set are well defined we had to perform some data augmentation in order to add some data variations and help the model improve its accuracy of prediction especially in our very limited training dataset of 95 images. Without this, the model would just learn to recognize the images of the train set and would overfit.
Data augmentation is made "on-the-fly", meaning that each transformation is randomly computed anew at each epoch of training (in fact, each time we go through the `__getitem__` of the `Dataset`), making sure that the model learns the distinct features of the chocolates and not the images. Unfortunately, this comes at the cost of an increased training time (1 minute of training per epoch instead of 10 seconds when doing offline data augmentation), but this greatly increased the generalization ability of the model.

We used several methods, here is a summary : 

- **Resizing**
	This transformation is the only one applied to both training and validation sets. It reduces every image from 6000x4000 to 576x384 (but everything can be changed in the `src/Constants.py` file and the rest of the code will adapt automatically). This reduction is usefull to save computational time during the training of the model. It also removes some noise and high frequency features, keeping only more general features.

- **Affine Transformations**  
  These geometric transformations help the model become invariant to position and orientation changes. It is important to note that we took care of discarding the chocolates' bounding boxes which would be out of bounds of the image due to the transformations as these would result in corrupted data. 
  - **Random Horizontal and Vertical Flip**: With a 50% chance, the image and its bounding boxes are flipped horizontally or vertically.
  - **Random Rotation**: Applies a random rotation (±`max_rotation` degrees) and updates the bounding boxes accordingly.
  - **Random Translation**: Shifts the image up to `max_translation` of its dimensions and adjusts bounding boxes to match.

- **Color Jittering**  
  Randomly modifies brightness, contrast, saturation, and hue to simulate different lighting conditions.

- **Gaussian / Box Blur**  
  With `blur_prob` (30 %) probability, applies either a Gaussian blur (60% chance) or a box blur (40%) with random radius.

- **Gaussian Noise + Salt & Pepper Noise**  
  With `noise_prob` probability, adds Gaussian noise with varying intensity and optionally adds salt-and-pepper noise (20% chance).


Here are some examples of images with the transforms applied : 

<p align="center">
  <img src="src/Images_report/img_10.png" width="30%" style="margin: 10px;" />
  <img src="src/Images_report/img_18.png" width="30%" style="margin: 10px;" />
  <img src="src/Images_report/img_21.png" width="30%" style="margin: 10px;" />
</p>

<p align="center"><em> <strong>Figure 1 </strong>: Here we have selected some images where the transformatiosn are clearly visible. The first image illustrates the Gaussian noise, the second the Salt & Pepper noise and the third one the Gaussian blur, each time with random affine transformations.</em></p>


## YOLO: main concepts

YOLO (You Only Look Once) is a family of object detection algorithms designed for real-time applications. Unlike traditional methods that process candidate regions in multiple steps, YOLO reframes object detection as a single regression problem, directly predicting bounding boxes and class probabilities from the entire image in one pass.

YOLO uses a neural network in order to simultaneously predict different features of the objects to detect. Indeed, the neural networks takes an image as input, and by several convolutional layers and reductions, it outputs a grid of a predefined size.

In each cell, the model should predict : the confidence of having an object to detect inside, the center coordinates of the object with respect to the cell, the width and height of the bounding box, and a vector of the size of the number of classes, that should ideally be a one-hot encoding vector predicting the right class. It is important to understand that the model predicts something in each cell, even if there is no object. In this case the predicted score should be low, and that is only during inference that we select only cells where the model is pretty sure that there is an object, for example we select all the cells that have a score higher than 0.7 or 0.8.

In our case, we implemented YOLOv2 from scratch. Unlike the general YOLO overview described earlier, YOLOv2 introduces the concept of **anchor boxes**, predefined bounding box shapes that the model uses to guide its predictions. These anchors allow the model to focus on refining the position and size of objects relative to typical bounding box shapes, rather than predicting box dimensions entirely from scratch. This significantly improves training efficiency and detection accuracy (https://arxiv.org/pdf/1612.08242).
To define the anchor boxes, we implemented a **K-Means clustering algorithm** on all the ground truth bounding boxes in our training set. This clustering helped us determine the most representative box shapes for our dataset.

<table style="width:100%; border:none; border-collapse:collapse;">
  <tr>
    <td style="width:50%; vertical-align:top; border:none; padding-right:10px;">
      <p>
        On the right picture we can observe the result of K-Means. This picture is created by the file
        <code>src/Anchors/anchor_calculator.py</code>.<br><br>
        We can observe all the ground truth boxes of the train set in blue, and the result of K-Means in red. The lengths are in pixels, with respect to the initial images size (6000x4000). We can observe that most of the boxes are squared, and sometimes they are very elongated, in the case of a vertical Amandina chocolate (A6) or horizontal (A7).<br><br>
        The code also provides the average Intersection-over-Union ratio (IoU) of the boxes of the ground truth with their closest anchor, which is 0.9415. This means that, on average, we can approximate any ground truth box by its closest anchor without losing much precision. In this way, the model will not have to create the boxes from scratch, but only align its prediction with one of 9 anchors. Actually, this is the biggest improvement with respect to YOLOv1.<br><br>
        To begin our implementation, we referred to a YOLOv3-from-scratch tutorial available <a href="https://www.geeksforgeeks.org/yolov3-from-scratch-using-pytorch/" target="_blank">here</a>. This article was particularly helpful in helping us understand key YOLO-related concepts and the overall architecture. The YOLOv3 implementation in the article includes <strong>multi-scale object detection</strong>, which is designed to detect objects of varying sizes more effectively. However, in our case, since all chocolates are relatively similar in size, we decided that incorporating multi-scale predictions would add unnecessary complexity without significant benefits.
      </p>
    </td>
    <td style="width:50%; vertical-align:top; border:none;">
      <img src="src/Anchors/output_anchors/anchors_visualization.png" width="100%">
      <p style="text-align:center;"><em><strong>Figure 2</strong>: The results from our K-means.</em></p>
    </td>
  </tr>
</table>

Therefore, we started from an existing codebase, preserved its overall structure, and removed everything related to multi-scale predictions (specific to later versions of YOLO). This adaptation to YOLOv2 was particularly challenging because many aspects of the original code were hard-coded. To proceed, we had to deeply analyze the code and understand it line by line, especially the data conversion and reshaping operations. Most of the functions were difficult to adapt, so we chose to rewrite them entirely, retaining only their names and high-level logic, such as the training loop structure. In the end, very little of the original code remains. We only kept a few low-level utility components, including: `iou` (to compute a ratio between 2 rectangles), `save_checkpoint`and `load_checkpoint` (to save and load a model) and some parts of the `Dataset` class and the `YOLOLoss` class. Moreover, the YOLO architecture usually takes squared images as input, but we thought that it would be a good idea to keep the ratio 3:2, so we had to adapt everything. Indeed, if we resized the images to squares, some chocolates would be distorded differently according to their orientation, making the model more difficult to learn to recognize them.

At the end, what is really important to understand for the next part is that the `__getitem__` method of the `Dataset` class returns two things : the image transformed with the data augmentation of the given index, and the associated target. To create the target, we find, for each object of the image, the best anchor, and put a score of 1 in the right cell of the associated grid (among the 9). In this cell we also put the center of the box, its dimensions, and the class. If there is no object in a cell, every variable is 0. The output target shape from `__getitem__` is important to match the output shape of the neural networks, enabling to compare them inside the loss.

Therefore, the target associated to one image corresponds to 9 grids superimposed, each corresponding to an anchor. Every cell is full of 0 except where the center of an object lies. In this cell, we put a 1 only in the grid corresponding to the closest anchor of the initial bounding box. This cell also contains the center coordinates and the dimension of the initial labeled box. We can see an example in Figure 3.

<p align="center">
    <img src="src/Images_report/Amandina_image_anchor.png" width="60%" style="margin: 10px;" />
</p>

<p align="center"><em>  

**Figure 3**: This example features the *Amandina* chocolate, which is relatively easy to detect due to its distinct shape compared to other chocolates. The image displayed corresponds to one of the 9 target grids, in this case, the grid associated with anchor **A7** (On **Figure 2**, we can see easily that the closest anchor for this chocolate is **A7**). We overlayed the initial photo on it for understanding purpose. Other grids are irrelevant for this object, as its shape will only activate this specific anchor. By consequent all the cells of the other grids are full of **0**. We observe that all cells in the **A7** grid have a value of **0**, except for the one inside which the center of the Amandina lies. The activated cell represents the location of the center of the anchor's bounding box, correctly identifying the presence of the object.
</em></p> 


## Model

In this section, we will present our neural network model. Even though this element is mandatory for everything to work, however its content could be changed as long as the input and the output are the same. For recall, the input is a batch of images of dimension (3, 384, 576), while its output is a batch whose each element consists of 9 grids 24x36 (or 12x18, depending on the argument `grid_ratio`). Each cell of the grids consists of `5+num_classes` values, with the 5 corresponding to the score of having the center of an object in the cell, the coordinates of the predicted center with respect to the cell, and the width and height of the predicted bounding box. The second part is a vector of dimension `num_classes` that represents a distribution of probabilities.

Our final model consists of 2 main bricks: a CNN block and a ScalePrediction block. Let's detail both of them.

#### CNNBlock:
A modular building block that applies:
- A convolutional layer (no bias)
- Batch normalization
- SiLU (Swish) activation

The convolutional layer applies spatial filtering to extract local features. The `bias=False` is intentional when followed by Batch Normalization, since BatchNorm already includes a learnable bias term (beta). The batch normalization normalizes the activations for each channel across the mini-batch : $\hat{x_i} = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ and $y_i = \gamma \hat{x_i} + \beta$, with $\gamma$ and $\beta$ learnable. It helps stabilize learning and improve convergence. Finallly the SiLU activation is a non-linearity of the form $SiLU(x)=x \cdot sigmoid(x)$. Without it, multiple linear layers are equivalent to once. Unlike ReLU, SiLU is smooth and non-monotonic. It often outperforms ReLU in vision tasks, avoiding "dying neuron" problems (where gradients are zero for negative inputs).

#### ScalePrediction
A prediction module that outputs bounding box predictions at a given scale:
- First applies a CNNBlock for feature refinement
- Then a `Conv2d` layer to produce `(num_anchors * (5 + num_classes))` channels of dimensions `(H,W)`
- The output is reshaped to `(B, num_anchors, H, W, 5 + num_classes)`

Each anchor predicts:
- 4 box coordinates (tx, ty, tw, th)
- 1 objectness score
- `num_classes` class logits that corresponds to a distribution of probabilities on the classes after applying a SoftMax.

### YOLO_Model
The main model, composed of:

**Backbone**: a series of CNNBlocks with periodic `stride=2` to downsample the input spatially. It reduces resolution from 1× to 1/16 of the input size, while expanding the number of channels progressively.

**Extra Downsampling**: if `grid_ratio == 32`, an additional CNNBlock pair downsamples the feature map to 1/32 resolution. If `grid_ratio == 16`, this stage is skipped.

**Head**: a lightweight convolutional head to further process features before prediction. It consists of two CNNBlocks with an intermediate Dropout layer. The dropout is a regularization technique used to prevent overfitting in neural networks. During training, dropout randomly sets a fraction `p` (0.1 in our case) of the input units to zero at each forward pass (while it is disabled during evaluation). This forces the network to avoid over-relying on any particular neuron and promotes redundancy in learned features.

**Prediction**: the final output block combines a CNNBlock and a `ScalePrediction` module.

The result is a tensor of shape: `(B, num_anchors, H_grid, W_grid, 5 + num_classes)`.

#### Hyperparameters
- `num_classes`: number of object classes (e.g. 13 for chocolate types)
- `num_anchors`: number of anchors (e.g. 9 from the K-Means)
- `grid_ratio`: defines ratio of the output resolution with the initial picture resolution (e.g. 16 or 32)
- `scale_factor`: scales the number of channels for lightweight variants

#### Weight Initialization (`init_yolo_weights`)
- All Conv2d layers are initialized with Kaiming Normal (He initialization)
- BatchNorm weights set to 1, biases to 0
- The final prediction layer has:
  - Objectness bias initialized to -4.6 (≈ 0.01 after sigmoid)
  - All other biases set to 0 (coordinates and class logits)
	
It is important to set the bias of the objectness around 0 post-sigmoid in order to avoid some false positives from the beginning, helping the model to converge.

### Important remarks on the model:

As we said before, the content of the model is important for the efficiency of the algorithm, but can be change easily if the input and output shapes remain the same. Therefore we changed several times during the project. We were limited to 12 M trainable weights, so we began intuitively with the biggest possible model around 11-12 M weights. However, we did not understand why our score remained so low, around 30% of F1 score on the test set. Then we added the noise in data augmentation, and were able to reach around 60% at our best. We got stuck around this score for a long time, until we add dropout in the model, and the model instantly reached around 80%. We understood that our model was too powerful for the tiny dataset we have. Indeed, the easiest way for the model to reduce the loss was to reconize the photos of the train set and place the chocolates accordingly: it was overfitting, even though we had a powerfull data augmentation. Therefore, the dropout corrected that but we thought that we should reduce the model in order to learn more general features.

Therefore, we kept the general structure of the model but reduced the number of layers, and added the argument `scale_factor` to automatically change the number of weights. Setting it to `0.5` leads to our current model, with `1 201 170` learnable parameters. We finally reached the baseline with this model.



## Loss

The loss is one of the most important elements in the project. It can be found in the `YOLOLoss` class. It makes the neural network converge to what we want. Indeed, through training iterations, the loss will decrease, but we have to define it very well such that its decrease has the wanted effects (in our case, we should have a score of 1 in the right cells, and the correct box dimensions with the correct class).

To achieve this, the implemented loss function is composed of four components: **L<sub>box</sub>**, **L<sub>no_obj</sub>**, **L<sub>obj</sub>**, and **L<sub>class</sub>**. Recall that the model outputs predictions on a `9` grids (for 9 anchors) of dimensions `24x36`. Each cell contains a confidence score indicating the likelihood that the center of an object is within the cell, the predicted center coordinates of the object relative to the cell, the width and height of the predicted bounding box, and a 13-dimensional vector representing the predicted class probabilities (after softmax) across the 13 possible categories.
Let's now take a closer look at the role of each loss component:

**L<sub>box</sub>**: for each cell of the 9 predicted grids, we apply a sigmoid on the predicted $x$ and $y$. On $w$ and $h$ of the 9 target grids, we apply a logarithm, then divide the value with the size of the anchor corresponding to the grid. We then apply a Mean Squared Error on $x$, $y$, $w$, $h$, in each cell where the target has an object. The goal of this is that when we obtain an output (9 grids), we should apply a sigmoid on $x$ and $y$ of each cell and they should be close to the corresponding target $x$ and $y$. For $w$ and $h$, we apply an exponential, multiply it with the anchor of the corresponding grid, and this should be close to the targeted $w$ and $h$. From the YOLOv2 paper (https://arxiv.org/pdf/1612.08242) we can understand that this operation is usefull to ensure positivity of $w$ and $h$ (with the exponential) and to learn the dimensions relatively to the anchors (multiplication with the anchors) ensuring to stabilize the learning.

<table style="width:100%; border:none; border-collapse:collapse;">
  <tr>
    <td style="width:50%; vertical-align:top; border:none;">
      <p> <strong>L<sub>no_obj</sub></strong>: for each cell of the 9 targeted grids that does not have an object, we compute the Binary Cross-Entropy (with logits) between the predicted score and the targeted score (that is 1). The BCEWithLogitsLoss applies a sigmoid internally.
			The goal is that, in regions where there is no object in the target, the predicted (post-sigmoid) objectness score should be close to 0. This component of the loss is critical for reducing the number of false positives. Without it, the model could trivially minimize the total loss by predicting objects in many locations, leading to over-detection, a behavior illustrated in Figure 4.
			The BCEWithLogitsLoss is particularly interesting for removing false positives because it penalizes much more a big prediction for a low ground truth than the other way.<br><br>
	  	<strong>L<sub>obj</sub></strong>: we apply the sigmoid on the predicted <i>x</i> and <i>y</i>. We also apply the exponential followed by the multiplication with the anchors on the predicted <i>w</i> and <i>h</i>. This way, we obtain 4 values to correctly define a predicted box. Next, we compute the Inter-over-Union between this rectangle and the targeted one. In the cells where the ground truth tells that there is an object, we compute the Mean Squared Error between the predicted score (post-sigmoid), and the Inter-over-Union ratio. It enables the predicted score post-sigmoid to be close to the Inter-over-Union score. This way, the predicted score is not 1 if the predicted box is not perfectly aligned with the target. This term basically manages the false negatives.
      </p>
    </td>
    <td style="width:50%; vertical-align:top; border:none;">
      <img src="src/Images_report/Loss_without_no_obj.png" width="100%"  id="k-means anchor">
			<p align="center"><em> <strong>Figure 4</strong>: The resulted model without the L<sub>no_obj</sub>, after convergence of the loss.</em></p>
    </td>
  </tr>
</table>

<strong>L<sub>class</sub></strong>: finally, for each cell of the predicted output, we take the vector of dimension 13, and apply a Cross-Entropy with the ground truth (an integer between 0 and 12). The Cross-Entropy penalizes if the logit of the correct class is too low. It makes the predicted 13-dimension vector converge to a one-hot encoding vector after being passed in a soft-max.


Finally, every term is just multiplied with an empirical weight. At first, we thought that we could just put the weight for L<sub>coord</sub> very low with respect to the others, because in this project we are interested only to count the chocolates, not localize them. However, it is more subtil than that. Indeed, the 4 losses are working together : if the coordinates are not well predicted, then it is difficult for the model to predict if there is an object, because the boundaries are not clear. Finally, we obtained a good model with the weights $\lambda_{box} = 1$, $\lambda_{obj} = 2$, $\lambda_{no\_obj} = 3$, $\lambda_{class} = 1$

## Post-processing

Once the model generates its raw predictions, we apply a post-processing pipeline to refine these outputs into meaningful detections. This step is essential both for visualization (`save_val_images()` in `src/utils.py`) and for generating the final submission file (`generate_submission_csv_with_nms()` in `src/Inference_test.py`).

The model outputs, for each grid cell and anchor box, a set of raw values including class scores (logits), objectness confidence, and bounding box parameters. To transform these into usable detections, we apply several key operations:

1. **From raw logits to representative values:**  
   Since the model outputs are raw logits, we apply the necessary transformations manually (same as in the loss) for each cell of each grid:
   - A **sigmoid** is applied to the predicted $x$ and $y$ values to constrain the center coordinates within each grid cell between 0 and 1.
   - An **exponential** is applied to the predicted $w$ and $h$ values, followed by a multiplication with the corresponding anchor dimensions to scale the box appropriately.
	- A **softmax** is applied to the vector of dimension `num_classes`, in order to obtain a distribution of probabilities on the classes.

2. **Score Filtering (Multiplying Class Probabilities with Objectness Confidence):**
	In each cell, after the previous operations were applied, we get a probability of having an object and a distribution of probabilities on its class. In our project, we are interesting of correctly classifying each object: it is not usefull to be sure at 90% that there is an object, if the distribution of probabilities is completely flat (unsure about its class). Therefore, we multiply the probability of having an object and the maximum probability inside the distribution among classes, to obtain the probability of having an object in the cell and being sure about its correct class. This gives is our final confidence score per cell.

3. **Thresholding:**  
   We discard all predictions whose final confidence score falls below a chosen threshold (e.g., 0.7). This keeps only the cells where the model is sure that it found the object and its class.

4. **Non-Maximum Suppression (NMS):**  
   Multiple close boxes may refer to the same object. NMS is used to remove redundant boxes by keeping only the most confident prediction among overlapping ones, based on a set IoU threshold.

5. **Final Box Selection and Counting:**  
   After applying NMS, we retain only the final set of detections for each image. These are then used to determine the number of detected objects per class and to generate the CSV submission file.

In short, post-processing ensures that raw model outputs are properly processed, filtered, and refined into clean, interpretable predictions. It prevents over-prediction and improves accuracy by removing low-confidence and overlapping boxes.


## Training

To train our YOLOv2 model, we built a modular and efficient training pipeline using PyTorch. The process begins by defining two datasets: one for training and one for validation. Both are handled using a custom `Dataset` class, which loads image–label pairs, applies transformations, and returns a tensor-formatted image along with a target tensor structured for YOLO.

The **target tensor** follows the YOLO format, with dimensions `(batch_size, num_anchors, grid_height, grid_width, 6)`, where each cell stores 6 values:
- An objectness score (1 or 0, as said in Figure 3),
- The center coordinates ($x$, $y$),
- The bounding box dimensions ($w$, $h$), and
- The class label, between 0 and 12.

Once the datasets are prepared, they are wrapped in PyTorch `DataLoader`s with appropriate shuffling and batching. The training data is augmented on-the-fly at each epoch, while the validation data is used as-is.

The model is initialized with a **24×36 output grid**, corresponding to a stride of 16 on 576×384 input images. We used the **Adam optimizer** with a small weight decay to improve generalization, and training is guided by our custom `YOLOLoss` function.


####  Per-Epoch Structure

Each training epoch consists of two main stages: training and validation.

1. **Training** is handled by the `train_one_epoch` function. For each batch of the epoch:
   - The image and target tensors are transferred to the GPU.
   - The image is give to the model that predicts the corresponding output.
   - The loss is computed by comparing these predictions to the ground truth using the `YOLOLoss` class.
   - Gradients are computed via backpropagation and the model weights are updated.
   - A running average of the loss is displayed using a progress bar.

   The output of this function is not only a trained model update, but also real-time feedback on how the model is learning throughout the epoch.

2. **Validation** is conducted immediately after each epoch using the `evaluate` function. The validation loss is tracked over time, and if it improves upon the best so far, we save a checkpoint of the model. This helps check for overfitting and ensures that the best-performing version of the model on unseen data is preserved.

Additionally, after each epoch, we generate visualizations of the model’s predictions on the validation set to qualitatively monitor its progress.


### Final Test Setup

Once training is complete, we prepare the test set using a `YOLOTestDataset` object. This dataset uses the same image preprocessing as the validation set but is reserved entirely for final inference. It ensures that performance is assessed on data the model has never seen during training or validation.




# Results & Conclusion

### Results

Our results have been quite satisfactory. We managed to surpass the benchmark on Kaggle, achieving a maximum score of **0.93342**, which confirms the effectiveness of our approach.

One of the key factors that influenced performance was the **conf_threshold** used during inference. We observed that when the threshold was too low, the model tended to produce false positives bounding boxes in areas where there were no chocolates, but with very low confidence scores. On the other hand, setting the threshold too high resulted in missed detections, especially for partially visible or less distinctive chocolates.
We attempted to identify an optimal threshold value, but since the model produces slightly different results across runs (due to data augmentation and weight initialization), it was difficult to fix a universally best value. After several experiments, we decided to set the confidence threshold to **75%**, which yielded the best and most consistent results on the Kaggle leaderboard.

( Include images of chocolates at several epochs, show some cases where the model doesnt work well maybe some amandina cases or jelly_white case , maybe also show a class that perfomed surprinsgly well, also could be interesting to specify whihc transform or which paramter in particular hekped us get good results and if know the results without those transforms or with differnet values specify it)



### Conclusion

In this project, we successfully implemented a chocolate detection system based on the YOLOv2 architecture. Our model is capable of identifying and localizing 13 different types of chocolates within high-resolution images, despite significant variability in background, lighting, and object occlusion.

One of the key challenges was working with a limited dataset, which we addressed through carefully designed data augmentation techniques and a custom annotation pipeline. We also implemented anchor-based object detection from scratch, using K-Means clustering to generate representative anchor boxes for our dataset.

Our custom loss function and training pipeline allowed the model to converge efficiently and achieve strong qualitative and quantitative results. The visual outputs demonstrate that the model correctly detects most chocolates in test images, and the validation performance shows promising generalization despite the small training set.

This project deepened our understanding of object detection principles, from label formatting and data preprocessing to anchor assignment and loss engineering. While our implementation focuses on a single-scale YOLOv2 model, the framework is flexible and could be extended to support multi-scale detection.

In future work, performance could be improved by:
- Incorporating additional training data,
- Exploring multi-scale architectures (like YOLOv3 or YOLOv5),
- Applying model ensembling or fine-tuning pretrained weights.

Overall, this project highlights the practical challenges and creative solutions involved in designing a deep learning-based object detection system tailored to a specialized task.
